> **Data note:** this notebook was run on a private clinical cohort that is not included in this repository, so it will not run as-is. The public diabetes dataset used in the paper is in `data/`; see `03_rail_d.ipynb` and `04_c_rail_d.ipynb` for runnable versions.

In [ ]:
from dotenv import load_dotenv, find_dotenv
import requests
import os
from langchain_openai import ChatOpenAI
import pandas as pd
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate
import math
from tqdm import tqdm
import json
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

In [ ]:
#all configurations

#EXCEL_PATH = "../data/Patients_toy.xlsx"       # adjust path if neede
EXCEL_PATH = "../data/PatientsF_with_slopes_new.xlsx"
SHEET = 0                                           # or a sheet name
MODEL = "gpt-4o-mini"   # any OpenAI chat model
TEMPERATURE = 0.0                                   # deterministic classification
TARGET_COL = "IEC_HS"                               # ground-truth label in your file
OUT_CSV = "../outputs/granite3-1-8B.csv"

In [ ]:
# OpenAI API access: put OPENAI_API_KEY in your environment or in a local .env file
# (see .env.example). Never commit your key.
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set")

In [ ]:
# strcutured output and getting the prompt

class IecHsPrediction(BaseModel):
    iec_hs: Literal[0, 1] = Field(..., description="Predicted IEC_HS class label (0 or 1).")
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Model self-reported confidence between 0 and 1."
    )
    rationale: str = Field(
        ..., description="Brief clinical reasoning behind the prediction."
    )

#parser = JsonOutputParser(pydantic_object=IecHsPrediction)
parser = PydanticOutputParser(pydantic_object=IecHsPrediction)

system_msg = """You are a careful clinical classifier. 
Given a single patient's anonymized features, predict whether IEC_HS = 0 or 1.
Only use the provided features—do not invent data.

Output MUST be valid JSON matching this schema:
- iec_hs: 0 or 1
- confidence: a number in [0, 1]
- rationale: short justification (1–3 sentences)
"""

human_template = """
Patient features (key → value) as JSON:
{patient_json}

Instructions:
1) Predict IEC_HS (0 or 1). 
2) Be conservative if evidence is weak.
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_msg), ("human", human_template)]
)

# IMPORTANT: inject the raw JSON schema as a *variable*, not as literal text
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [ ]:
llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        max_retries=2,
        max_tokens=2048,
        api_key=api_key,
    )

In [ ]:
chain = prompt | llm | parser

In [ ]:
def row_to_features(row: pd.Series) -> dict:
    """Convert a row to a JSON-serializable dict, handling NaN/None cleanly."""
    out = {}
    for k in feature_cols:
        v = row[k]
        if isinstance(v, float) and math.isnan(v):
            out[k] = None
        else:
            # cast numpy types to native
            if hasattr(v, "item"):
                try:
                    v = v.item()
                except Exception:
                    pass
            out[k] = v
    return out


# Load data
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET)

# Build feature set: everything except the target
feature_cols = [c for c in df.columns if c != TARGET_COL]

In [ ]:
results = []


for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring patients"):
    patient_features = row_to_features(row)
    # Safety: ensure we never leak the ground-truth label to the model
    assert TARGET_COL not in patient_features


    pred: IecHsPrediction = chain.invoke({
        "patient_json": json.dumps(patient_features, ensure_ascii=False)
    })
    #print(f"prediction is {pred['iec_hs']}")
    print(f"prediction is {pred.iec_hs} with confidence {pred.confidence}")
    results.append({
            "row_index": idx,
            "iec_hs_pred": int(pred.iec_hs),
            "confidence": float(pred.confidence),
            "rationale": pred.rationale,
            "iec_hs_true": row.get(TARGET_COL, None)
        })

In [ ]:
out_df = pd.DataFrame(results)
print(out_df.columns.tolist())
# Optional: simple accuracy if labels exist
if TARGET_COL in df.columns and out_df["iec_hs_pred"].notna().any():

    acc = (out_df["iec_hs_pred"] == out_df["iec_hs_true"]).mean()
    print(f"Accuracy on labeled rows: {acc:.3f}")

#print(out_df)
out_df.to_csv(OUT_CSV, index=False)

In [ ]:
out_df = pd.DataFrame(results)

# Keep only rows where both are non-null
valid = out_df["iec_hs_pred"].notna() & out_df["iec_hs_true"].notna()
y_true = out_df.loc[valid, "iec_hs_true"].astype(int)
y_pred = out_df.loc[valid, "iec_hs_pred"].astype(int)

# Overall metrics (positive class = 1)
accuracy   = accuracy_score(y_true, y_pred)
precision1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
recall1    = recall_score(y_true, y_pred,    pos_label=1, zero_division=0)
f1_pos     = f1_score(y_true, y_pred,        pos_label=1, zero_division=0)

# Negative-class metrics (label = 0)
precision0 = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
recall0    = recall_score(y_true, y_pred,    pos_label=0, zero_division=0)
f1_neg     = f1_score(y_true, y_pred,        pos_label=0, zero_division=0)

# Macro F1 (unweighted mean of per-class F1s)
f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"Accuracy:     {accuracy:.3f}")
print(f"Pos P/R/F1:   P={precision1:.3f}  R={recall1:.3f}  F1={f1_pos:.3f}")
print(f"Neg P/R/F1:   P={precision0:.3f}  R={recall0:.3f}  F1={f1_neg:.3f}")
print(f"Macro F1:     {f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("\nConfusion Matrix (rows = true, cols = pred):")
print("         Pred 0   Pred 1")
print(f"True 0   {cm[0,0]:6}   {cm[0,1]:6}")
print(f"True 1   {cm[1,0]:6}   {cm[1,1]:6}")